In [7]:
!pip install -q fastapi uvicorn requests python-dotenv pydantic nest-asyncio pyngrok

In [8]:
import os

os.makedirs("ai_agent", exist_ok=True)

files = [
    "ai_agent/__init__.py",
    "ai_agent/agent.py",
    "ai_agent/tools.py",
    "ai_agent/config.py",
    "ai_agent/main.py",
]

for file in files:
    open(file, "w").close()

print("Project structure created.")

Project structure created.


In [10]:
!pip install -q google-genai

In [13]:
pip install -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 54.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.18.0
    Uninstalling transformers-5.18.0:
      Successfully uninstalled transformers-5.18.0


In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen3-8B"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("Model loaded successfully!")

Loading tokenizer...


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded successfully!


In [15]:
prompt = """
Explain what an AI agent is.
Give me a simple explanation and one real-world example.
"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        do_sample=True
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

<think>
Okay, the user is asking for an explanation of what an AI agent is, with a simple explanation and a real-world example. Let me start by recalling the definition. An AI agent is a system that can perceive its environment and take actions to maximize some notion of cumulative reward. But that's a bit technical. I need to simplify it.

Maybe start by saying it's a program that can make decisions or take actions on its own. Like a helper that does tasks without human input. Then, think of examples. The classic ones are virtual assistants like Siri or Alexa. But wait, the user wants a real-world example. Let me think of something more tangible. Maybe a recommendation system, like Netflix suggesting movies. Wait, but that's more of a service. Or a self-driving car? That's a good example because it's an AI agent interacting with the environment in real-time. Let me verify: a self-driving car perceives its surroundings through sensors, processes that data, and makes decisions like stee

In [16]:
def generate_response(prompt, max_new_tokens=300):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response

In [17]:
answer = generate_response(
    "What is the difference between an AI assistant and an autonomous AI agent?"
)

print(answer)

<think>
Okay, so I need to figure out the difference between an AI assistant and an autonomous AI agent. Let me start by recalling what I know about AI. 

First, an AI assistant. I've heard terms like Siri, Alexa, or Google Assistant. They're tools that help users with tasks like setting reminders, answering questions, or controlling smart devices. They respond to user commands, right? So they're interactive, but they rely on the user to give them specific instructions. They don't make decisions on their own; they follow the user's input.

Now, autonomous AI agents. I'm not as familiar with that term, but maybe it's like a more advanced version? Maybe something that can operate independently without constant human input. Like a robot or a system that can make decisions on its own. For example, self-driving cars or automated trading systems. These systems might have their own goals and can take actions to achieve them without needing a person to tell them what to do each time.

Wait, bu

In [18]:
import math


def calculator(expression: str):
    """
    Safely evaluate basic mathematical expressions.
    """

    allowed = {
        "abs": abs,
        "round": round,
        "min": min,
        "max": max,
        "pow": pow,
        "sqrt": math.sqrt,
        "sin": math.sin,
        "cos": math.cos,
        "tan": math.tan,
        "pi": math.pi,
    }

    try:
        result = eval(
            expression,
            {"__builtins__": {}},
            allowed
        )

        return str(result)

    except Exception as e:
        return f"Calculator error: {e}"

In [19]:
print(calculator("25 * 18"))
print(calculator("sqrt(144)"))
print(calculator("100 / 4 + 15"))

450
12.0
40.0


In [22]:
AGENT_SYSTEM_PROMPT = """
You are an autonomous AI agent.

You have access to the following tools:

1. calculator
   Description: Performs mathematical calculations.

When you need to use a tool, respond exactly in this format:

TOOL:
tool_name
INPUT:
tool_input

When you have enough information to answer the user, respond:

FINAL:
your answer

Do not invent tool results.
"""

In [26]:
import re
import torch


def clean_agent_response(response):
    """
    Remove Qwen's internal reasoning section.
    """

    # Remove <think>...</think>
    response = re.sub(
        r"<think>.*?</think>",
        "",
        response,
        flags=re.DOTALL
    )

    return response.strip()


def agent_llm(prompt):

    messages = [
        {
            "role": "system",
            "content": AGENT_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.1,
            do_sample=True
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return clean_agent_response(response)

In [29]:
def run_agent(user_task, max_steps=5):

    conversation = f"""
User request:

{user_task}
"""

    for step in range(max_steps):

        print(f"\n--- Agent Step {step + 1} ---")

        response = agent_llm(conversation)

        print("Agent:")
        print(response)

        # --------------------------------
        # FINAL ANSWER
        # --------------------------------

        if response.startswith("FINAL:"):

            final_answer = response[
                len("FINAL:"):
            ].strip()

            return final_answer

        # --------------------------------
        # TOOL REQUEST
        # --------------------------------

        if "TOOL:" in response:

            tool_match = re.search(
                r"TOOL:\s*(.+)",
                response
            )

            input_match = re.search(
                r"INPUT:\s*(.+)",
                response,
                re.DOTALL
            )

            if not tool_match or not input_match:

                print("Could not parse tool request.")

                conversation += f"""

Your previous response was invalid:

{response}

Use EXACTLY one of these formats:

TOOL:
calculator
INPUT:
25 * 10

OR:

FINAL:
your answer
"""

                continue

            tool_name = tool_match.group(1).strip()
            tool_input = input_match.group(1).strip()

            # --------------------------------
            # CALCULATOR
            # --------------------------------

            if tool_name == "calculator":

                result = calculator(tool_input)

                print("\nTool:")
                print("calculator")

                print("\nInput:")
                print(tool_input)

                print("\nResult:")
                print(result)

                conversation += f"""

You requested the calculator tool.

Expression:
{tool_input}

Calculator result:
{result}

Now use this result.

If you are finished, respond:

FINAL:
<answer>

Do not request the same calculation again.
"""

                continue

            else:

                return f"Unknown tool: {tool_name}"

        # --------------------------------
        # INVALID RESPONSE
        # --------------------------------

        conversation += f"""

Your previous response was:

{response}

You must either request a tool using:

TOOL:
calculator
INPUT:
<expression>

or provide:

FINAL:
<answer>
"""

    return "Agent reached the maximum number of steps."

In [27]:
AGENT_SYSTEM_PROMPT = """
You are an autonomous AI agent.

You have access to this tool:

calculator
Description:
Performs mathematical calculations.

IMPORTANT RULES:

1. Do not output your internal reasoning.
2. Do not use <think> tags in your response.
3. If you need the calculator, output ONLY:

TOOL:
calculator
INPUT:
<mathematical expression>

4. Do not output FINAL when requesting a tool.

5. After receiving the tool result, provide:

FINAL:
<answer>

6. If no tool is required, immediately provide:

FINAL:
<answer>
"""

In [30]:
result = run_agent(
    "What is 1250 multiplied by 48, and then divided by 6?"
)

print("\n====================")
print("FINAL ANSWER")
print("====================")
print(result)


--- Agent Step 1 ---
Agent:
<think>
Okay, let's see. The user wants to know what is 1250 multiplied by 48 and then divided by 6. Hmm, so first I need to calculate 1250 times 48. Let me do that step by step. 

Wait, maybe I can simplify the calculation before multiplying. Since dividing by 6 after multiplying might be a big number, perhaps I can divide 48 by 6 first? Let me check. If I divide 48 by 6, that gives 8. So then the problem becomes 1250 multiplied by 8. That seems easier. 

Let me confirm that approach. Multiplying 1250 by 8. Well, 1000 times 8 is 8000, and 250 times 8 is 2000. Adding those together gives 8000 + 2000 = 10,000. Wait, that's a nice round number. So the result would be 10,000? Let me verify with the original numbers to make sure I didn't make a mistake.

Alternatively, if I do 1250 * 48 first. Let's break that down. 1250 * 40 is 50,000, and 1250 * 8 is 10,

--- Agent Step 2 ---
Agent:
TOOL:
calculator
INPUT:
1250 * 48 / 6

Tool:
calculator

Input:
1250 * 48 / 6

In [31]:
!pip install -q ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 73.6 MB/s eta 0:00:00


In [32]:
from ddgs import DDGS


def web_search(query: str, max_results=5):

    try:
        results = DDGS().text(
            query,
            max_results=max_results
        )

        formatted = []

        for result in results:

            formatted.append({
                "title": result.get("title", ""),
                "url": result.get("href", ""),
                "snippet": result.get("body", "")
            })

        return formatted

    except Exception as e:

        return {
            "error": str(e)
        }

In [33]:
results = web_search("latest AI agent developments")

for result in results:
    print(result["title"])
    print(result["url"])
    print(result["snippet"])
    print("-" * 80)

Latest AI developments 2026: Models, agentic AI, infrastructure
https://www.netguru.com/blog/latest-ai-developments-2026
Aug 20, 2026 · Latest AI developments 2026: frontier model releases, agentic AI in production, infrastructure shifts, and security risks. See what actually changes your roadmap.
--------------------------------------------------------------------------------
The best AI agent frameworks in 2026 - langchain.com
https://www.langchain.com/resources/ai-agent-frameworks
Jun 6, 2026 · We reviewed 7 AI agent frameworks across orchestration, observability, and production readiness. See how LangGraph, CrewAI, Microsoft Agent Framework, and others compare.
--------------------------------------------------------------------------------
The Best AI Agents in 2026: Tools and Frameworks Compared
https://www.datacamp.com/blog/best-ai-agents
May 28, 2026 · Discover 2026's best AI agents. Compare frameworks, no-code tools, enterprise platforms, and get step-by-step guidance to choos

In [34]:
AGENT_SYSTEM_PROMPT = """
You are an autonomous AI research agent.

You have access to these tools.

TOOL 1: calculator

Description:
Performs mathematical calculations.

Use:

TOOL:
calculator
INPUT:
<mathematical expression>


TOOL 2: web_search

Description:
Searches the internet for current information.

Use:

TOOL:
web_search
INPUT:
<search query>


IMPORTANT RULES:

1. Do not output internal reasoning.
2. Do not use <think> tags.
3. Use web_search when the user asks for:
   - current information
   - latest information
   - recent events
   - information you don't know
   - research
4. Use calculator for mathematical calculations.
5. You can use multiple tools if necessary.
6. Never invent tool results.
7. After receiving tool results, decide what to do next.
8. When you have enough information, respond:

FINAL:
<answer>

9. When requesting a tool, output ONLY the tool request.
"""

In [35]:
def run_agent(user_task, max_steps=6):

    conversation = f"""
User request:

{user_task}
"""

    for step in range(max_steps):

        print(f"\n{'='*60}")
        print(f"AGENT STEP {step + 1}")
        print(f"{'='*60}")

        response = agent_llm(conversation)

        print("\nAgent:")
        print(response)

        # --------------------------------
        # FINAL
        # --------------------------------

        if response.startswith("FINAL:"):

            return response[
                len("FINAL:"):
            ].strip()

        # --------------------------------
        # TOOL
        # --------------------------------

        if "TOOL:" in response:

            tool_match = re.search(
                r"TOOL:\s*(.+)",
                response
            )

            input_match = re.search(
                r"INPUT:\s*(.+)",
                response,
                re.DOTALL
            )

            if not tool_match or not input_match:

                conversation += f"""

Your previous response was invalid:

{response}

Use:

TOOL:
tool_name
INPUT:
input

OR:

FINAL:
answer
"""

                continue

            tool_name = tool_match.group(1).strip()
            tool_input = input_match.group(1).strip()

            # =================================
            # CALCULATOR
            # =================================

            if tool_name == "calculator":

                result = calculator(tool_input)

                print("\nTool: calculator")
                print("Input:", tool_input)
                print("Result:", result)

                conversation += f"""

Tool used: calculator

Input:
{tool_input}

Result:
{result}

Continue solving the user's request.

If finished:

FINAL:
<answer>
"""

            # =================================
            # WEB SEARCH
            # =================================

            elif tool_name == "web_search":

                results = web_search(tool_input)

                print("\nTool: web_search")
                print("Query:", tool_input)

                formatted_results = ""

                for i, result in enumerate(results, 1):

                    formatted_results += f"""
Result {i}:

Title:
{result.get('title', '')}

URL:
{result.get('url', '')}

Snippet:
{result.get('snippet', '')}

"""

                print(formatted_results)

                conversation += f"""

Tool used: web_search

Search query:
{tool_input}

Search results:
{formatted_results}

Use these results to answer the user's request.

Do not invent information.

If you need more information, you can use web_search again.

If finished:

FINAL:
<answer>
"""

            else:

                return f"Unknown tool: {tool_name}"

        else:

            conversation += f"""

Your previous response did not follow the required format.

Response:
{response}

You must either request a tool:

TOOL:
tool_name
INPUT:
input

or provide:

FINAL:
answer
"""

    return "Agent reached maximum number of steps."

In [36]:
result = run_agent(
    "What is 7500 multiplied by 24 and divided by 8?"
)

print("\nFINAL:")
print(result)


AGENT STEP 1

Agent:
<think>
Okay, the user is asking for 7500 multiplied by 24 and then divided by 8. Let me break this down. First, I need to calculate 7500 times 24. Then take that result and divide it by 8. 

Hmm, maybe I can simplify the calculation. Instead of doing 7500*24 first, perhaps I can divide 24 by 8 first since division is the inverse of multiplication. Wait, 24 divided by 8 is 3. So then the problem becomes 7500 multiplied by 3. That seems easier. Let me check if that's mathematically valid. 

Yes, because (7500 * 24) / 8 is the same as 7500 * (24/8) due to the associative property of multiplication and division. So 24 divided by 8 is indeed 3. Then 7500 times 3 is 22,500. 

But to make sure I didn't make a mistake, maybe I should verify using the calculator tool. Let me use the calculator step by step. First, 7500 * 24 equals 180,000. Then dividing 180,000 by 8 gives 22,500. Yep, that matches. So the answer should

AGENT STEP 2

Agent:
TOOL:
calculator
INPUT:
7500 * 

In [37]:
result = run_agent(
    "Search the web and tell me about the latest developments in AI agents."
)

print("\nFINAL:")
print(result)


AGENT STEP 1

Agent:
TOOL:
web_search
INPUT:
latest developments in AI agents 2023

Tool: web_search
Query: latest developments in AI agents 2023

Result 1:

Title:
Evolution of AI Agents: 2023-2026 Timeline & History | metacto

URL:
https://www.metacto.com/blogs/evolution-of-ai-agents-2023-to-2026

Snippet:
From AutoGPT to Devin, Manus, ChatGPT Agent, and Claude Computer Use: trace the full evolution of AI agents 2023-2026, the milestones that mattered, and what comes next.


Result 2:

Title:
The Evolution of AI Agents: From AutoGPT to Production (2023-2026)

URL:
https://thedatapraxis.com/blog/agent-evolution-timeline/

Snippet:
Executive Briefing What this covers: A chronological narrative of AI agent development from the AutoGPT explosion (March 2023) through production maturity (2026), organized into four distinct phases. Who should read it: Engineering leaders, product managers, and architects evaluating where their agent initiatives sit on the maturity curve.


Result 3:

Titl

In [38]:
import io
import contextlib
import traceback


def python_tool(code: str):
    """
    Execute Python code inside the current Colab runtime.

    Intended for trusted development/testing only.
    """

    stdout = io.StringIO()

    try:

        with contextlib.redirect_stdout(stdout):

            exec(
                code,
                {
                    "__builtins__": __builtins__,
                    "math": __import__("math"),
                    "statistics": __import__("statistics"),
                }
            )

        output = stdout.getvalue()

        if not output:
            output = "Code executed successfully with no output."

        return output

    except Exception:

        return traceback.format_exc()

In [39]:
result = python_tool("""
numbers = [10, 20, 30, 40, 50]

average = sum(numbers) / len(numbers)

print("Average:", average)
""")

print(result)

Average: 30.0



In [40]:
result = python_tool("""
import math

numbers = [12, 18, 25, 31, 42]

mean = sum(numbers) / len(numbers)

variance = sum(
    (x - mean) ** 2
    for x in numbers
) / len(numbers)

std = math.sqrt(variance)

print("Mean:", mean)
print("Standard deviation:", std)
""")

print(result)

Mean: 25.6
Standard deviation: 10.403845442911962



In [41]:
AGENT_SYSTEM_PROMPT = """
You are an autonomous AI agent.

You have access to three tools.

========================
TOOL 1: calculator
========================

Description:
Performs mathematical calculations.

Format:

TOOL:
calculator
INPUT:
<mathematical expression>


========================
TOOL 2: web_search
========================

Description:
Searches the internet for current information.

Format:

TOOL:
web_search
INPUT:
<search query>


========================
TOOL 3: python_tool
========================

Description:
Executes Python code for data analysis,
calculations and computation.

Format:

TOOL:
python_tool
INPUT:
<python code>


========================
RULES
========================

1. Do not output internal reasoning.

2. Do not use <think> tags.

3. Use calculator for simple mathematical calculations.

4. Use python_tool for:
   - data analysis
   - statistics
   - numerical computation
   - Python calculations
   - data processing

5. Use web_search for:
   - current information
   - recent information
   - research
   - information you do not know

6. You can use multiple tools.

7. Never invent tool results.

8. After receiving a tool result, decide what to do next.

9. When finished, respond:

FINAL:
<answer>

10. When requesting a tool, output ONLY:

TOOL:
tool_name
INPUT:
input
"""

In [44]:
import re


def run_agent(user_task, max_steps=6):

    conversation = f"""
User request:

{user_task}

Remember:
- Use a tool when necessary.
- After receiving a tool result, continue the task.
- When finished, output ONLY:
FINAL:
<answer>
"""

    for step in range(max_steps):

        print(f"\n{'=' * 60}")
        print(f"AGENT STEP {step + 1}")
        print(f"{'=' * 60}")

        response = agent_llm(conversation)

        print("\nAgent:")
        print(response)

        # ==================================================
        # FINAL ANSWER
        # ==================================================

        final_match = re.search(
            r"FINAL:\s*(.*)",
            response,
            re.DOTALL
        )

        if final_match:

            final_answer = final_match.group(1).strip()

            # Remove accidental tool request after FINAL
            if "TOOL:" in final_answer:
                final_answer = final_answer.split("TOOL:")[0].strip()

            return final_answer

        # ==================================================
        # TOOL REQUEST
        # ==================================================

        tool_match = re.search(
            r"TOOL:\s*([a-zA-Z_]+)",
            response
        )

        if not tool_match:

            conversation += f"""

Your previous response was invalid.

You responded:

{response}

You MUST respond using exactly one of these formats:

TOOL:
calculator
INPUT:
<expression>

OR:

TOOL:
web_search
INPUT:
<query>

OR:

TOOL:
python_tool
INPUT:
<python code>

OR:

FINAL:
<answer>
"""

            continue

        tool_name = tool_match.group(1).strip()

        # ==================================================
        # EXTRACT INPUT
        # ==================================================

        input_match = re.search(
            r"INPUT:\s*(.*)",
            response,
            re.DOTALL
        )

        if not input_match:

            conversation += f"""

Your tool request is missing INPUT.

Your response:

{response}

Correct format:

TOOL:
tool_name
INPUT:
input
"""

            continue

        tool_input = input_match.group(1).strip()

        # Remove accidental FINAL from tool input
        if "FINAL:" in tool_input:

            tool_input = tool_input.split("FINAL:")[0].strip()

        # ==================================================
        # CALCULATOR
        # ==================================================

        if tool_name == "calculator":

            result = calculator(tool_input)

            print("\nTool: calculator")
            print("Input:")
            print(tool_input)

            print("\nResult:")
            print(result)

            conversation += f"""

Tool used: calculator

Expression:
{tool_input}

Calculator result:
{result}

Use this result to continue solving the user's request.

If finished, respond ONLY:

FINAL:
<answer>
"""

        # ==================================================
        # WEB SEARCH
        # ==================================================

        elif tool_name == "web_search":

            results = web_search(tool_input)

            print("\nTool: web_search")
            print("Query:")
            print(tool_input)

            formatted_results = ""

            if isinstance(results, dict):

                formatted_results = str(results)

            else:

                for i, result in enumerate(results, 1):

                    formatted_results += f"""
RESULT {i}

Title:
{result.get("title", "")}

URL:
{result.get("url", "")}

Snippet:
{result.get("snippet", "")}

"""

            print("\nSearch Results:")
            print(formatted_results)

            conversation += f"""

Tool used: web_search

Search query:
{tool_input}

Search results:

{formatted_results}

Use the search results to answer the user's request.

Important:
- Do not invent facts.
- If the information is insufficient, perform another search.
- If enough information is available, provide the final answer.

When finished:

FINAL:
<answer>
"""

        # ==================================================
        # PYTHON
        # ==================================================

        elif tool_name == "python_tool":

            result = python_tool(tool_input)

            print("\nTool: python_tool")

            print("\nCode:")
            print(tool_input)

            print("\nPython Result:")
            print(result)

            conversation += f"""

Tool used: python_tool

Python code:

{tool_input}

Python execution result:

{result}

Use the Python result to continue solving the user's request.

If finished:

FINAL:
<answer>
"""

        # ==================================================
        # UNKNOWN TOOL
        # ==================================================

        else:

            conversation += f"""

The requested tool does not exist:

{tool_name}

Available tools:

calculator
web_search
python_tool

Please choose one of these tools or provide:

FINAL:
<answer>
"""

    return "Agent reached maximum number of steps."

In [46]:
result = run_agent(
    "Calculate 8750 * 36 / 9"
)

print("\nFINAL ANSWER:")
print(result)


AGENT STEP 1

Agent:
<think>
Okay, let's see. The user wants me to calculate 8750 multiplied by 36 and then divided by 9. Hmm, I can approach this step by step. First, maybe I should do the multiplication part. So 8750 times 36. Let me check if I can use the calculator tool for this. The rules say to use the calculator for simple math calculations. But wait, multiplying 8750 by 36 might be a bit complex. Alternatively, maybe I can simplify the calculation before using the tool. Let me think: dividing by 9 after multiplying by 36. Wait, 36 divided by 9 is 4. Oh, right! So maybe I can simplify the problem first. Instead of multiplying by 36 and then dividing by 9, I can just multiply 8750 by 4. That would make the calculation easier. Let me verify that. Yes, because 36 divided by 9 is 4, so the expression becomes 8750 * 4. That's simpler. So then, 8750 times 4. Let me calculate that. 8000*4 is 32,000, and 750*4 is 3,000. Adding those together gives 35,000. Wait, is that right? Wait, 8



In [47]:
result = run_agent("""
Use Python to calculate the average, minimum,
maximum and standard deviation of:

[12, 25, 31, 44, 56, 72, 81]
""")

print("\nFINAL ANSWER:")
print(result)


AGENT STEP 1

Agent:
<think>
Okay, let's see. The user wants me to calculate the average, minimum, maximum, and standard deviation of the list [12, 25, 31, 44, 56, 72, 81]. Hmm, I need to use Python for this. Oh right, there's a python_tool available. Let me think about how to approach this.

First, I should use the python_tool to run the calculations. The list is given, so I can input it into Python. The steps are: calculate the average (mean), min, max, and standard deviation. For standard deviation, I need to compute the mean first, then subtract it from each number, square the results, take the average of those squares, and then take the square root. But maybe the Python code can handle that with built-in functions.

Wait, the python_tool can execute code, so I can write a script that does all these calculations. Let me draft the code. Let's assign the list to a variable, say 'numbers'. Then compute the mean using sum(numbers)/len(numbers). The min and max are straightforward with

In [48]:
result = run_agent("""
Use Python to analyze these numbers:

[15, 23, 31, 42, 55, 61, 72]

Calculate:
1. Mean
2. Median
3. Standard deviation
4. Minimum
5. Maximum

Explain the results.
""")

print("\n========================")
print("FINAL ANSWER")
print("========================")

print(result)


AGENT STEP 1

Agent:
<think>
Okay, let's tackle this problem. The user wants me to analyze the list of numbers [15, 23, 31, 42, 55, 61, 72] by calculating the mean, median, standard deviation, minimum, and maximum. Then I need to explain the results.

First, I should check which tools are available. The python_tool is suitable for data analysis tasks like this. The calculator tool is for simple math, but since there are multiple statistical calculations involved, using Python would be more efficient.

I'll start by using the python_tool to compute each of the required statistics. Let me think about the steps:

1. **Mean**: Sum all numbers and divide by the count.
2. **Median**: Sort the list (though it's already sorted) and find the middle value. Since there are 7 numbers, the 4th one is the median.
3. **Standard Deviation**: Calculate the mean, then the squared differences from the mean, average those squared differences (variance), and take the square root.
4. **Minimum and Maximum*

In [49]:
result = run_agent("""
Search the web for the latest NVIDIA GPU architecture.
Then use Python to organize the main GPU architectures
you find into a simple comparison.
""")

print("\n========================")
print("FINAL ANSWER")
print("========================")

print(result)


AGENT STEP 1

Agent:
<think>
Okay, let's tackle this user request. First, they want me to search the web for the latest NVIDIA GPU architecture. So I should start by using the web_search tool. The query should be something like "latest NVIDIA GPU architecture" to get the most up-to-date information.

Once I get the search results, I need to extract the main GPU architectures mentioned. Let's say the search results talk about architectures like Hopper (H100), Ada Lovelace (A100), and maybe some others. Wait, actually, the latest as of 2023 is Hopper, but I should verify that. The web search will confirm the current architecture.

After gathering the architectures, the next step is to use Python to organize them into a comparison. I'll need to structure the data, maybe in a dictionary or a list of tuples, with each architecture's name, release year, key features, and performance metrics. Then, I can use Python to format this data into a readable table or a structured comparison. I shoul

In [50]:
TOOLS = {
    "calculator": calculator,
    "web_search": web_search,
    "python_tool": python_tool
}

In [51]:
print(TOOLS.keys())

dict_keys(['calculator', 'web_search', 'python_tool'])


In [52]:
def execute_tool(tool_name, tool_input):

    if tool_name not in TOOLS:

        return {
            "success": False,
            "error": f"Unknown tool: {tool_name}"
        }

    try:

        tool_function = TOOLS[tool_name]

        result = tool_function(tool_input)

        return {
            "success": True,
            "tool": tool_name,
            "input": tool_input,
            "result": result
        }

    except Exception as e:

        return {
            "success": False,
            "tool": tool_name,
            "error": str(e)
        }

In [53]:
result = execute_tool(
    "calculator",
    "1250 * 8"
)

print(result)

{'success': True, 'tool': 'calculator', 'input': '1250 * 8', 'result': '10000'}


In [54]:
TOOL_DESCRIPTIONS = """
AVAILABLE TOOLS

1. calculator
Description:
Perform mathematical calculations.

Input:
A mathematical expression.

Example:
1250 * 48 / 6


2. web_search
Description:
Search the internet for current information.

Input:
A search query.

Example:
latest AI agent developments


3. python_tool
Description:
Execute Python code for computation and data analysis.

Input:
Python code.

Example:

numbers = [10, 20, 30]
print(sum(numbers) / len(numbers))
"""

In [55]:
AGENT_SYSTEM_PROMPT = f"""
You are an autonomous AI agent.

You can use the following tools:

{TOOL_DESCRIPTIONS}

RULES:

1. Do not expose internal reasoning.

2. Use a tool when it is useful.

3. Never invent tool results.

4. You may use multiple tools.

5. After receiving a tool result, continue solving the task.

6. When you have completed the task, respond:

FINAL:
<answer>

7. When using a tool:

TOOL:
<tool_name>
INPUT:
<tool_input>
"""

In [56]:
class AgentMemory:

    def __init__(self, max_messages=10):

        self.messages = []

        self.max_messages = max_messages

    def add(self, role, content):

        self.messages.append({
            "role": role,
            "content": content
        })

        # Keep memory bounded
        if len(self.messages) > self.max_messages:

            self.messages = self.messages[
                -self.max_messages:
            ]

    def get_context(self):

        context = ""

        for message in self.messages:

            context += f"""
{message["role"].upper()}:

{message["content"]}

"""

        return context

    def clear(self):

        self.messages = []

In [57]:
memory = AgentMemory()

memory.add(
    "user",
    "My name is Basil."
)

memory.add(
    "assistant",
    "Nice to meet you, Basil."
)

memory.add(
    "user",
    "I am building an AI agent."
)

print(memory.get_context())


USER:

My name is Basil.


ASSISTANT:

Nice to meet you, Basil.


USER:

I am building an AI agent.




In [59]:
import re


class Agent:

    def __init__(self, max_steps=6):

        self.max_steps = max_steps

        # Conversation memory
        self.memory = AgentMemory()

        # Track tools used during the current task
        self.tools_used = []


    # ==========================================================
    # BUILD CONVERSATION
    # ==========================================================

    def build_conversation(self, user_task):

        memory_context = self.memory.get_context()

        conversation = f"""
{AGENT_SYSTEM_PROMPT}

==============================
CONVERSATION MEMORY
==============================

{memory_context}

==============================
CURRENT USER REQUEST
==============================

{user_task}

Remember:

- Use a tool when necessary.
- Never invent tool results.
- After receiving a tool result, continue solving the task.
- You may use multiple tools.
- When finished output:

FINAL:
<answer>
"""

        return conversation


    # ==========================================================
    # PARSE TOOL REQUEST
    # ==========================================================

    def parse_tool_request(self, response):

        tool_match = re.search(
            r"TOOL:\s*([a-zA-Z_]+)",
            response,
            re.IGNORECASE
        )

        if not tool_match:
            return None, None

        tool_name = tool_match.group(1).strip()

        input_match = re.search(
            r"INPUT:\s*(.*)",
            response,
            re.DOTALL | re.IGNORECASE
        )

        if not input_match:
            return tool_name, None

        tool_input = input_match.group(1).strip()

        # Remove accidental FINAL
        if "FINAL:" in tool_input:
            tool_input = tool_input.split("FINAL:")[0].strip()

        return tool_name, tool_input


    # ==========================================================
    # EXTRACT FINAL ANSWER
    # ==========================================================

    def parse_final_answer(self, response):

        final_match = re.search(
            r"FINAL:\s*(.*)",
            response,
            re.DOTALL | re.IGNORECASE
        )

        if not final_match:
            return None

        answer = final_match.group(1).strip()

        # Remove accidental tool request
        if "TOOL:" in answer:
            answer = answer.split("TOOL:")[0].strip()

        return answer


    # ==========================================================
    # EXECUTE TOOL
    # ==========================================================

    def execute(self, tool_name, tool_input):

        if tool_name not in TOOLS:

            return {
                "success": False,
                "error": f"Unknown tool: {tool_name}"
            }

        try:

            result = execute_tool(
                tool_name,
                tool_input
            )

            self.tools_used.append(tool_name)

            return result

        except Exception as e:

            return {
                "success": False,
                "error": str(e)
            }


    # ==========================================================
    # RUN AGENT
    # ==========================================================

    def run(self, user_task):

        self.tools_used = []

        conversation = self.build_conversation(user_task)

        # Save user message
        self.memory.add(
            "user",
            user_task
        )


        for step in range(self.max_steps):

            print("\n" + "=" * 60)
            print(f"AGENT STEP {step + 1}")
            print("=" * 60)


            # ==================================================
            # CALL LLM
            # ==================================================

            response = agent_llm(conversation)

            print("\nAgent:")
            print(response)


            # ==================================================
            # CHECK FINAL ANSWER
            # ==================================================

            final_answer = self.parse_final_answer(response)

            if final_answer:

                print("\nFinal Answer:")
                print(final_answer)

                # Save assistant response
                self.memory.add(
                    "assistant",
                    final_answer
                )

                return {
                    "success": True,
                    "answer": final_answer,
                    "tools_used": self.tools_used,
                    "steps": step + 1
                }


            # ==================================================
            # PARSE TOOL REQUEST
            # ==================================================

            tool_name, tool_input = self.parse_tool_request(
                response
            )


            # ==================================================
            # INVALID RESPONSE
            # ==================================================

            if not tool_name:

                conversation += f"""

Your previous response was invalid.

You responded:

{response}

You MUST use exactly one of these formats:

TOOL:
calculator
INPUT:
<expression>

OR

TOOL:
web_search
INPUT:
<query>

OR

TOOL:
python_tool
INPUT:
<python code>

OR

FINAL:
<answer>
"""

                continue


            # ==================================================
            # MISSING INPUT
            # ==================================================

            if not tool_input:

                conversation += f"""

Your tool request is missing INPUT.

Your response:

{response}

Correct format:

TOOL:
{tool_name}
INPUT:
<tool input>
"""

                continue


            # ==================================================
            # EXECUTE TOOL
            # ==================================================

            print("\nTool:", tool_name)

            print("\nInput:")
            print(tool_input)


            result = self.execute(
                tool_name,
                tool_input
            )


            print("\nTool Result:")
            print(result)


            # ==================================================
            # SEND RESULT BACK TO LLM
            # ==================================================

            conversation += f"""

==============================
TOOL RESULT
==============================

Tool:

{tool_name}

Input:

{tool_input}

Result:

{result}

==============================

Continue solving the user's request.

Do not invent information.

If another tool is required, use it.

If the task is complete:

FINAL:
<answer>
"""


        # ======================================================
        # MAX STEPS REACHED
        # ======================================================

        return {
            "success": False,
            "answer": "Agent reached maximum number of steps.",
            "tools_used": self.tools_used,
            "steps": self.max_steps
        }

In [60]:
agent = Agent(max_steps=6)

In [61]:
result = agent.run(
    "What is 1250 multiplied by 48 and then divided by 6?"
)

print(result)


AGENT STEP 1

Agent:
TOOL:
calculator
INPUT:
1250 * 48 / 6

FINAL:
<answer>10000</answer>

Final Answer:
<answer>10000</answer>
{'success': True, 'answer': '<answer>10000</answer>', 'tools_used': [], 'steps': 1}


In [62]:
result = agent.run(
    "Search the web for the latest developments in AI agents and give me a short summary."
)

print(result)


AGENT STEP 1

Agent:
TOOL:
web_search
INPUT:
latest AI agent developments

Tool: web_search

Input:
latest AI agent developments

Tool Result:
{'success': True, 'tool': 'web_search', 'input': 'latest AI agent developments', 'result': [{'title': 'Latest AI developments 2026: Models, agentic AI, infrastructure', 'url': 'https://www.netguru.com/blog/latest-ai-developments-2026', 'snippet': 'Aug 20, 2026 · Latest AI developments 2026: frontier model releases, agentic AI in production, infrastructure shifts, and security risks. See what actually changes your roadmap.'}, {'title': 'The best AI agent frameworks in 2026 - langchain.com', 'url': 'https://www.langchain.com/resources/ai-agent-frameworks', 'snippet': 'Jun 6, 2026 · We reviewed 7 AI agent frameworks across orchestration, observability, and production readiness. See how LangGraph, CrewAI, Microsoft Agent Framework, and others compare.'}, {'title': 'The Best AI Agents in 2026: Tools and Frameworks Compared', 'url': 'https://www.data

In [63]:
result = agent.run(
    """
Use Python to calculate the average of:

12, 25, 37, 48, 51, 63

Then explain the result.
"""
)

print(result)


AGENT STEP 1

Agent:
<think>
Okay, let's see. The user wants me to calculate the average of these numbers: 12, 25, 37, 48, 51, 63. Hmm, right. So first, I need to add all those numbers together and then divide by how many there are. Let me check how many numbers there are. Let me count: 12 is the first, 25 second, 37 third, 48 fourth, 51 fifth, 63 sixth. So six numbers in total.

Wait, maybe I should use the python_tool for this. The user mentioned using Python to calculate the average. The example given was using a list and summing then dividing by the length. So I can create a list of those numbers, sum them, and then divide by 6. Let me make sure the numbers are correct. The numbers are 12, 25, 37, 48, 51, 63. Let me add them step by step. 12 + 25 is 37. Then 37 + 37 is 74. 74 + 48 is 122. 122 + 51 is 173. 173 + 63 is 236. Wait, is that right? Let me check again. 1

AGENT STEP 2

Agent:
<think>
Okay, let's tackle this problem. The user wants the average of the numbers 12, 25, 37, 4

In [64]:
!pip install -q fastapi uvicorn nest-asyncio pyngrok

In [65]:
from fastapi import FastAPI
from pydantic import BaseModel
import nest_asyncio

nest_asyncio.apply()

app = FastAPI(
    title="Local Qwen AI Agent",
    description="AI Agent powered by Qwen3-8B with Calculator, Web Search and Python tools",
    version="1.0.0"
)

In [66]:
class AgentRequest(BaseModel):
    task: str

In [67]:
class AgentResponse(BaseModel):
    success: bool
    answer: str
    tools_used: list
    steps: int

In [68]:
@app.get("/health")
def health():

    return {
        "status": "healthy",
        "model": "Qwen/Qwen3-8B",
        "agent": "online"
    }

In [69]:
@app.post("/agent", response_model=AgentResponse)
def run_agent_api(request: AgentRequest):

    result = agent.run(request.task)

    return result

In [74]:
%%writefile main.py

from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(
    title="Local Qwen AI Agent",
    description="AI Agent powered by Qwen3-8B",
    version="1.0.0"
)


class AgentRequest(BaseModel):
    task: str


class AgentResponse(BaseModel):
    success: bool
    answer: str
    tools_used: list
    steps: int


@app.get("/health")
def health():

    return {
        "status": "healthy",
        "model": "Qwen/Qwen3-8B",
        "agent": "online"
    }


@app.post("/agent", response_model=AgentResponse)
def run_agent_api(request: AgentRequest):

    result = agent.run(request.task)

    return result

Writing main.py


In [75]:
import subprocess
import sys
import time

server = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "main:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(3)

print("Uvicorn process started.")

Uvicorn process started.


In [76]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/health"
)

print(response.status_code)
print(response.json())

200
{'status': 'healthy', 'model': 'Qwen/Qwen3-8B', 'agent': 'online'}


In [81]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/health"
)

print("STATUS:", response.status_code)
print("CONTENT TYPE:", response.headers.get("content-type"))
print("RAW RESPONSE:")
print(response.text)

STATUS: 200
CONTENT TYPE: application/json
RAW RESPONSE:
{"status":"healthy","model":"Qwen/Qwen3-8B","agent":"online"}


In [82]:
%%writefile llm.py

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re

MODEL_NAME = "Qwen/Qwen3-8B"

print("Loading Qwen3-8B...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("Qwen3-8B loaded.")


def agent_llm(prompt, max_new_tokens=500):

    messages = [
        {
            "role": "system",
            "content": """
You are an autonomous AI agent.

You can use tools when necessary.

When you need a tool, respond EXACTLY:

TOOL:
tool_name
INPUT:
tool input

Available tools:

calculator
web_search
python_tool

When the task is complete, respond EXACTLY:

FINAL:
answer

Do not expose internal reasoning.
"""
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            do_sample=True
        )

    generated_tokens = outputs[
        0
    ][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    # Remove Qwen reasoning blocks
    response = re.sub(
        r"<think>.*?</think>",
        "",
        response,
        flags=re.DOTALL
    )

    return response.strip()

Writing llm.py


In [83]:
%%writefile tools.py

import math
import io
import contextlib
import traceback

from ddgs import DDGS


# ==========================================================
# CALCULATOR
# ==========================================================

def calculator(expression: str):

    allowed = {
        "abs": abs,
        "round": round,
        "min": min,
        "max": max,
        "pow": pow,
        "sqrt": math.sqrt,
        "sin": math.sin,
        "cos": math.cos,
        "tan": math.tan,
        "pi": math.pi
    }

    try:

        result = eval(
            expression,
            {"__builtins__": {}},
            allowed
        )

        return str(result)

    except Exception as e:

        return f"Calculator error: {e}"


# ==========================================================
# WEB SEARCH
# ==========================================================

def web_search(query: str, max_results=5):

    try:

        results = DDGS().text(
            query,
            max_results=max_results
        )

        formatted = []

        for result in results:

            formatted.append({
                "title": result.get("title", ""),
                "url": result.get("href", ""),
                "snippet": result.get("body", "")
            })

        return formatted

    except Exception as e:

        return {
            "error": str(e)
        }


# ==========================================================
# PYTHON TOOL
# ==========================================================

def python_tool(code: str):

    stdout = io.StringIO()

    try:

        with contextlib.redirect_stdout(stdout):

            exec(
                code,
                {
                    "__builtins__": __builtins__,
                    "math": __import__("math"),
                    "statistics": __import__("statistics")
                }
            )

        output = stdout.getvalue()

        if not output:

            output = "Code executed successfully with no output."

        return output

    except Exception:

        return traceback.format_exc()


# ==========================================================
# TOOL REGISTRY
# ==========================================================

TOOLS = {

    "calculator": calculator,

    "web_search": web_search,

    "python_tool": python_tool

}


def execute_tool(tool_name, tool_input):

    if tool_name not in TOOLS:

        return {
            "success": False,
            "error": f"Unknown tool: {tool_name}"
        }

    try:

        result = TOOLS[tool_name](tool_input)

        return {
            "success": True,
            "tool": tool_name,
            "input": tool_input,
            "result": result
        }

    except Exception as e:

        return {
            "success": False,
            "tool": tool_name,
            "error": str(e)
        }

Writing tools.py


In [84]:
%%writefile memory.py

class AgentMemory:

    def __init__(self, max_messages=10):

        self.messages = []

        self.max_messages = max_messages


    def add(self, role, content):

        self.messages.append({
            "role": role,
            "content": content
        })

        if len(self.messages) > self.max_messages:

            self.messages = self.messages[
                -self.max_messages:
            ]


    def get_context(self):

        context = ""

        for message in self.messages:

            context += f"""

{message["role"].upper()}:

{message["content"]}

"""

        return context


    def clear(self):

        self.messages = []

Writing memory.py


In [85]:
%%writefile agent.py

import re

from llm import agent_llm
from tools import execute_tool
from memory import AgentMemory


class Agent:

    def __init__(self, max_steps=6):

        self.max_steps = max_steps

        self.memory = AgentMemory()

        self.tools_used = []


    # ======================================================
    # PARSE FINAL
    # ======================================================

    def parse_final(self, response):

        match = re.search(
            r"FINAL:\s*(.*)",
            response,
            re.DOTALL | re.IGNORECASE
        )

        if not match:

            return None

        answer = match.group(1).strip()

        if "TOOL:" in answer:

            answer = answer.split("TOOL:")[0].strip()

        return answer


    # ======================================================
    # PARSE TOOL
    # ======================================================

    def parse_tool(self, response):

        tool_match = re.search(
            r"TOOL:\s*([a-zA-Z_]+)",
            response,
            re.IGNORECASE
        )

        if not tool_match:

            return None, None

        tool_name = tool_match.group(1).strip()

        input_match = re.search(
            r"INPUT:\s*(.*)",
            response,
            re.DOTALL | re.IGNORECASE
        )

        if not input_match:

            return tool_name, None

        tool_input = input_match.group(1).strip()

        if "FINAL:" in tool_input:

            tool_input = tool_input.split(
                "FINAL:"
            )[0].strip()

        return tool_name, tool_input


    # ======================================================
    # RUN
    # ======================================================

    def run(self, user_task):

        self.tools_used = []

        self.memory.add(
            "user",
            user_task
        )

        conversation = f"""

Previous conversation:

{self.memory.get_context()}

Current task:

{user_task}

Use tools when necessary.

When finished:

FINAL:
<answer>
"""


        for step in range(self.max_steps):

            response = agent_llm(
                conversation
            )

            print(
                f"\n--- Agent Step {step + 1} ---"
            )

            print(response)


            # ==============================================
            # FINAL
            # ==============================================

            final_answer = self.parse_final(
                response
            )

            if final_answer:

                self.memory.add(
                    "assistant",
                    final_answer
                )

                return {

                    "success": True,

                    "answer": final_answer,

                    "tools_used":
                        self.tools_used,

                    "steps":
                        step + 1

                }


            # ==============================================
            # TOOL
            # ==============================================

            tool_name, tool_input = self.parse_tool(
                response
            )


            if not tool_name:

                conversation += """

Your response was invalid.

You must use:

TOOL:
tool_name
INPUT:
input

OR:

FINAL:
answer
"""

                continue


            if not tool_input:

                conversation += """

The tool request is missing INPUT.

Please provide:

TOOL:
tool_name
INPUT:
input
"""

                continue


            # ==============================================
            # EXECUTE
            # ==============================================

            result = execute_tool(
                tool_name,
                tool_input
            )

            self.tools_used.append(
                tool_name
            )

            print(
                "\nTool:",
                tool_name
            )

            print(
                "\nResult:",
                result
            )


            # ==============================================
            # FEED RESULT BACK
            # ==============================================

            conversation += f"""

Tool used:

{tool_name}

Tool input:

{tool_input}

Tool result:

{result}

Continue solving the task.

If complete:

FINAL:
<answer>
"""


        return {

            "success": False,

            "answer":
                "Agent reached maximum number of steps.",

            "tools_used":
                self.tools_used,

            "steps":
                self.max_steps

        }

Writing agent.py


In [86]:
%%writefile main.py

from fastapi import FastAPI
from pydantic import BaseModel

from agent import Agent


app = FastAPI(
    title="Local Qwen AI Agent",
    description="Autonomous AI Agent powered by Qwen3-8B",
    version="1.0.0"
)


# ==========================================================
# AGENT
# ==========================================================

agent = Agent(
    max_steps=6
)


# ==========================================================
# REQUEST
# ==========================================================

class AgentRequest(BaseModel):

    task: str


# ==========================================================
# RESPONSE
# ==========================================================

class AgentResponse(BaseModel):

    success: bool

    answer: str

    tools_used: list

    steps: int


# ==========================================================
# HEALTH
# ==========================================================

@app.get("/health")
def health():

    return {

        "status": "healthy",

        "model": "Qwen/Qwen3-8B",

        "agent": "online"

    }


# ==========================================================
# AGENT API
# ==========================================================

@app.post(
    "/agent",
    response_model=AgentResponse
)
def run_agent_api(
    request: AgentRequest
):

    return agent.run(
        request.task
    )

Overwriting main.py


In [87]:
server.terminate()

In [89]:
import subprocess
import sys
import time

server = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "main:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000"
    ]
)

time.sleep(5)

print("FastAPI server started.")

FastAPI server started.


In [92]:
import subprocess
import sys

server = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "main:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000"
    ],
    stdout=None,
    stderr=None
)

print("Started server PID:", server.pid)

Started server PID: 33888


In [93]:
import requests

r = requests.get(
    "http://127.0.0.1:8000/health"
)

print(r.status_code)
print(r.json())

200
{'status': 'healthy', 'model': 'Qwen/Qwen3-8B', 'agent': 'online'}


In [94]:
r = requests.post(
    "http://127.0.0.1:8000/agent",
    json={
        "task": "Calculate 1250 * 48 / 6"
    }
)

print("STATUS:", r.status_code)
print("RESPONSE:")
print(r.json())

STATUS: 200
RESPONSE:
{'success': True, 'answer': '10000', 'tools_used': [], 'steps': 1}


In [95]:
response = requests.post(
    "http://127.0.0.1:8000/agent",
    json={
        "task": "Search the web for recent developments in autonomous AI agents and summarize them."
    }
)

print(response.status_code)

result = response.json()

print("\nANSWER:")
print(result["answer"])

print("\nTOOLS:")
print(result["tools_used"])

print("\nSTEPS:")
print(result["steps"])

200

ANSWER:
<answer>Recent developments in autonomous AI agents in 2023 include advancements in reinforcement learning for complex decision-making, improved natural language processing for human-AI collaboration, and enhanced ethical frameworks to ensure responsible AI behavior. Notable innovations involve more efficient training methods, better adaptability in dynamic environments, and increased integration of AI agents in real-world applications like healthcare and autonomous vehicles. Researchers are also focusing on improving transparency and reducing biases in AI systems.</answer>

TOOLS:
[]

STEPS:
1


In [100]:
!wget -q -O cloudflared-linux-amd64.deb \
https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb

In [101]:
!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package cloudflared.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.10.0) ...
Setting up cloudflared (2026.10.0) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [107]:
import subprocess
import re
import time

tunnel = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

for _ in range(20):
    line = tunnel.stdout.readline()

    if line:
        print(line.strip())

    if "trycloudflare.com" in line:
        break

2026-10-10T18:34:54Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-10T18:34:54Z INF Requesting new quick Tunnel on trycloudflare.com...


In [108]:
import time

time.sleep(3)

for _ in range(30):
    line = tunnel.stdout.readline()
    if line:
        print(line.strip())

2026-10-10T18:34:57Z INF +--------------------------------------------------------------------------------------------+
2026-10-10T18:34:57Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-10T18:34:57Z INF |  https://john-intake-sheffield-manufacturing.trycloudflare.com                             |
2026-10-10T18:34:57Z INF +--------------------------------------------------------------------------------------------+
2026-10-10T18:34:57Z INF Cannot determine default configuration path. No file [config.yml config.yaml] in [~/.cloudflared ~/.cloudflare-warp ~/cloudflare-warp /etc/cloudflared /usr/local/etc/cloudflared]
2026-10-10T18:34:57Z INF Version 2026.10.0 (Checksum d33ff2d14475178d2012c2c56beba87389ac5ded27649519f198a7d3134a99db)
2026-10-10T18:34:57Z INF GOOS: linux, GOVersion: go1.26.8, GoArch: amd64
2026-10-10T18:34:57Z INF Settings: map[ha-connections:1 protocol:quic url:http://127.0.0.1:8000]
2026-10-10T18:34:57Z INF cl

In [109]:
PUBLIC_URL = "https://john-intake-sheffield-manufacturing.trycloudflare.com"

In [110]:
import requests

r = requests.get(
    f"{PUBLIC_URL}/health"
)

print(r.status_code)
print(r.text)

200
{"status":"healthy","model":"Qwen/Qwen3-8B","agent":"online"}


In [111]:
r = requests.post(
    f"{PUBLIC_URL}/agent",
    json={
        "task": "Calculate 1250 * 48 / 6 and explain the result."
    },
    timeout=180
)

print("STATUS:", r.status_code)
print("RESPONSE:")
print(r.json())

STATUS: 200
RESPONSE:
{'success': True, 'answer': '<answer>The calculation of 1250 * 48 / 6 is as follows:\n\nFirst, multiply 1250 by 48:\n1250 * 48 = 60,000\n\nThen, divide the result by 6:\n60,000 / 6 = 10,000\n\nTherefore, the result of 1250 * 48 / 6 is 10,000.</answer>', 'tools_used': [], 'steps': 1}


In [112]:
!mkdir -p ai-agent
!mkdir -p ai-agent

In [113]:
%%writefile ai-agent/llm.py

import re
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM


MODEL_NAME = "Qwen/Qwen3-8B"


print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


print("Loading Qwen3-8B...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("Qwen3-8B loaded successfully.")


def agent_llm(prompt: str, max_new_tokens: int = 400):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.2,
            top_p=0.9
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    # Remove Qwen reasoning block
    response = re.sub(
        r"<think>.*?</think>",
        "",
        response,
        flags=re.DOTALL
    )

    return response.strip()

Writing ai-agent/llm.py


In [114]:
%%writefile ai-agent/memory.py

class AgentMemory:

    def __init__(self, max_messages=10):

        self.messages = []

        self.max_messages = max_messages


    def add(self, role, content):

        self.messages.append(
            {
                "role": role,
                "content": content
            }
        )

        if len(self.messages) > self.max_messages:

            self.messages = self.messages[
                -self.max_messages:
            ]


    def get_context(self):

        context = ""

        for message in self.messages:

            context += f"""
{message["role"].upper()}:

{message["content"]}

"""

        return context


    def clear(self):

        self.messages = []

Writing ai-agent/memory.py


In [115]:
%%writefile ai-agent/tools.py

import math

from ddgs import DDGS


# ==========================================
# CALCULATOR
# ==========================================

def calculator(expression: str):

    allowed = {

        "abs": abs,
        "round": round,
        "min": min,
        "max": max,
        "pow": pow,
        "sqrt": math.sqrt,

        "sin": math.sin,
        "cos": math.cos,
        "tan": math.tan,

        "pi": math.pi
    }

    try:

        result = eval(
            expression,
            {
                "__builtins__": {}
            },
            allowed
        )

        return str(result)

    except Exception as e:

        return f"Calculator error: {e}"


# ==========================================
# WEB SEARCH
# ==========================================

def web_search(query: str, max_results=5):

    try:

        results = DDGS().text(
            query,
            max_results=max_results
        )

        formatted = []

        for result in results:

            formatted.append(
                {
                    "title": result.get(
                        "title",
                        ""
                    ),

                    "url": result.get(
                        "href",
                        ""
                    ),

                    "snippet": result.get(
                        "body",
                        ""
                    )
                }
            )

        return formatted

    except Exception as e:

        return {
            "error": str(e)
        }


# ==========================================
# TOOL REGISTRY
# ==========================================

TOOLS = {

    "calculator": calculator,

    "web_search": web_search
}


# ==========================================
# TOOL EXECUTOR
# ==========================================

def execute_tool(tool_name, tool_input):

    if tool_name not in TOOLS:

        return {
            "success": False,
            "error": f"Unknown tool: {tool_name}"
        }

    try:

        result = TOOLS[tool_name](
            tool_input
        )

        return {

            "success": True,

            "tool": tool_name,

            "input": tool_input,

            "result": result
        }

    except Exception as e:

        return {

            "success": False,

            "tool": tool_name,

            "error": str(e)
        }

Writing ai-agent/tools.py


In [116]:
%%writefile ai-agent/agent.py

import re

from llm import agent_llm

from memory import AgentMemory

from tools import execute_tool


TOOL_DESCRIPTIONS = """

AVAILABLE TOOLS

1. calculator

Description:
Perform mathematical calculations.

Input:
A mathematical expression.

Example:

1250 * 48 / 6


2. web_search

Description:
Search the internet for current information.

Input:
A search query.

Example:

latest AI agent developments
"""


AGENT_SYSTEM_PROMPT = f"""

You are an autonomous AI agent.

You can use the following tools:

{TOOL_DESCRIPTIONS}


RULES:

1. Do not expose internal reasoning.

2. Use a tool when it is useful.

3. Never invent tool results.

4. You may use multiple tools.

5. After receiving a tool result,
continue solving the task.

6. When completed respond:

FINAL:
<answer>


7. When using a tool respond:

TOOL:
<tool_name>

INPUT:
<tool_input>

"""


class Agent:

    def __init__(self, max_steps=6):

        self.max_steps = max_steps

        self.memory = AgentMemory()

        self.tools_used = []


    # ======================================
    # BUILD PROMPT
    # ======================================

    def build_conversation(self, user_task):

        memory_context = self.memory.get_context()

        return f"""

{AGENT_SYSTEM_PROMPT}


==============================
CONVERSATION MEMORY
==============================

{memory_context}


==============================
CURRENT USER REQUEST
==============================

{user_task}


Remember:

- Use tools when necessary.
- Never invent results.
- You may use multiple tools.
- Do not expose reasoning.

If finished:

FINAL:
<answer>

"""


    # ======================================
    # PARSE TOOL
    # ======================================

    def parse_tool_request(self, response):

        tool_match = re.search(
            r"TOOL:\s*([a-zA-Z_]+)",
            response,
            re.IGNORECASE
        )

        if not tool_match:

            return None, None

        tool_name = tool_match.group(1).strip()

        input_match = re.search(
            r"INPUT:\s*(.*)",
            response,
            re.DOTALL | re.IGNORECASE
        )

        if not input_match:

            return tool_name, None

        tool_input = input_match.group(1).strip()

        if "FINAL:" in tool_input:

            tool_input = tool_input.split(
                "FINAL:"
            )[0].strip()

        return tool_name, tool_input


    # ======================================
    # PARSE FINAL
    # ======================================

    def parse_final_answer(self, response):

        final_match = re.search(
            r"FINAL:\s*(.*)",
            response,
            re.DOTALL | re.IGNORECASE
        )

        if not final_match:

            return None

        answer = final_match.group(1).strip()

        if "TOOL:" in answer:

            answer = answer.split(
                "TOOL:"
            )[0].strip()

        return answer


    # ======================================
    # RUN AGENT
    # ======================================

    def run(self, user_task):

        self.tools_used = []

        conversation = self.build_conversation(
            user_task
        )

        self.memory.add(
            "user",
            user_task
        )


        for step in range(self.max_steps):

            response = agent_llm(
                conversation
            )

            print(
                f"\n===== AGENT STEP {step + 1} ====="
            )

            print(response)


            # --------------------------
            # FINAL ANSWER
            # --------------------------

            final_answer = self.parse_final_answer(
                response
            )

            if final_answer:

                self.memory.add(
                    "assistant",
                    final_answer
                )

                return {

                    "success": True,

                    "answer": final_answer,

                    "tools_used": self.tools_used,

                    "steps": step + 1
                }


            # --------------------------
            # TOOL REQUEST
            # --------------------------

            tool_name, tool_input = \
                self.parse_tool_request(
                    response
                )


            if not tool_name:

                conversation += f"""

Your previous response was invalid.

You MUST respond using:

TOOL:
<tool_name>

INPUT:
<tool_input>

OR:

FINAL:
<answer>

"""

                continue


            if not tool_input:

                conversation += f"""

Your tool request is missing INPUT.

Correct format:

TOOL:
{tool_name}

INPUT:
<tool_input>

"""

                continue


            # --------------------------
            # EXECUTE TOOL
            # --------------------------

            result = execute_tool(
                tool_name,
                tool_input
            )

            self.tools_used.append(
                tool_name
            )


            print(
                "\nTOOL:",
                tool_name
            )

            print(
                "\nRESULT:",
                result
            )


            # --------------------------
            # FEED RESULT BACK TO LLM
            # --------------------------

            conversation += f"""

==============================
TOOL RESULT
==============================

Tool:

{tool_name}


Result:

{result}


==============================

Continue solving the user's request.

Do not invent information.

If another tool is required,
use it.

If the task is complete:

FINAL:
<answer>

"""


        return {

            "success": False,

            "answer":
            "Agent reached maximum number of steps.",

            "tools_used":
            self.tools_used,

            "steps":
            self.max_steps
        }

Writing ai-agent/agent.py


In [117]:
%%writefile ai-agent/main.py

from fastapi import FastAPI

from pydantic import BaseModel

from agent import Agent


app = FastAPI(

    title="Local Qwen AI Agent",

    description=
    "Autonomous AI Agent powered by Qwen3-8B",

    version="1.0.0"
)


agent = Agent(
    max_steps=6
)


class AgentRequest(BaseModel):

    task: str


class AgentResponse(BaseModel):

    success: bool

    answer: str

    tools_used: list

    steps: int


@app.get("/")

def root():

    return {

        "service":
        "Local Qwen AI Agent",

        "model":
        "Qwen/Qwen3-8B",

        "status":
        "online"
    }


@app.get("/health")

def health():

    return {

        "status":
        "healthy",

        "model":
        "Qwen/Qwen3-8B",

        "agent":
        "online"
    }


@app.post(
    "/agent",
    response_model=AgentResponse
)

def run_agent(
    request: AgentRequest
):

    result = agent.run(
        request.task
    )

    return result

Writing ai-agent/main.py


In [118]:
%%writefile ai-agent/requirements.txt

torch
transformers
accelerate
bitsandbytes
sentencepiece
fastapi
uvicorn
pydantic
ddgs

Writing ai-agent/requirements.txt


In [119]:
%%writefile ai-agent/Dockerfile

FROM nvidia/cuda:12.8.1-cudnn-runtime-ubuntu24.04

WORKDIR /app


RUN apt-get update && \
    apt-get install -y \
    python3 \
    python3-pip \
    git && \
    rm -rf /var/lib/apt/lists/*


COPY requirements.txt .


RUN pip3 install --break-system-packages \
    -r requirements.txt


COPY . .


EXPOSE 8000


CMD [
    "python3",
    "-m",
    "uvicorn",
    "main:app",
    "--host",
    "0.0.0.0",
    "--port",
    "8000"
]

Writing ai-agent/Dockerfile


In [120]:
!find ai-agent -maxdepth 1 -type f -print

ai-agent/llm.py
ai-agent/tools.py
ai-agent/main.py
ai-agent/agent.py
ai-agent/memory.py
ai-agent/Dockerfile
ai-agent/requirements.txt


In [121]:
!docker --version
!nvidia-smi

/bin/bash: line 1: docker: command not found
Sat Oct 10 18:43:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P0             57W /  400W |   33205MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+--

In [122]:
!zip -r local-qwen-agent.zip ai-agent

  adding: ai-agent/ (stored 0%)
  adding: ai-agent/llm.py (deflated 55%)
  adding: ai-agent/tools.py (deflated 74%)
  adding: ai-agent/main.py (deflated 59%)
  adding: ai-agent/agent.py (deflated 73%)
  adding: ai-agent/memory.py (deflated 64%)
  adding: ai-agent/Dockerfile (deflated 40%)
  adding: ai-agent/requirements.txt (deflated 23%)


In [123]:
from google.colab import files

files.download("local-qwen-agent.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [124]:
!apt-get update -qq
!apt-get install -y -qq docker.io

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Preconfiguring packages ...
Selecting previously unselected package netcat-openbsd.
(Reading database ... 127336 files and directories currently installed.)
Preparing to unpack .../00-netcat-openbsd_1.226-1ubuntu2_amd64.deb ...
Unpacking netcat-openbsd (1.226-1ubuntu2) ...
Selecting previously unselected package apparmor.
Preparing to unpack .../01-apparmor_4.0.1really4.0.1-0ubuntu0.24.04.8_amd64.deb ...
Unpacking apparmor (4.0.1really4.0.1-0ubuntu0.24.04.8) ...
Selecting previously unselected package libip4tc2:amd64.
Preparing to unpack .../02-libip4tc2_1.8.10-3ubuntu2_amd64.deb ...
Unpacking libip4tc2:amd64 (1.8.10-3ubuntu2) ...
Selecting previously unselected package libip6tc2:amd64.
Preparing to unpack .../03-libip6tc2_1.8.10-3ubuntu2_amd64.deb ...
Unpacking libip6tc2:amd64 (1.8.10-3ubuntu2) ...


In [125]:
!docker --version

Docker version 29.1.3, build 29.1.3-0ubuntu3~24.04.2


In [128]:
!service docker start

docker: unrecognized service


In [129]:
!docker info

Client:
 Version:    29.1.3
 Context:    default
 Debug Mode: false
 Plugins:
  trust: Manage trust on Docker images (Docker Inc.)
    Version:  29.1.3
    Path:     /usr/libexec/docker/cli-plugins/docker-trust

Server:
failed to connect to the docker API at unix:///var/run/docker.sock; check if the path is correct and if the daemon is running: dial unix /var/run/docker.sock: connect: no such file or directory


In [130]:
!dockerd > /tmp/dockerd.log 2>&1 &

In [131]:
import time
time.sleep(10)

In [132]:
!tail -50 /tmp/dockerd.log

time="2026-10-10T18:48:28.377713909Z" level=info msg="loading plugin" id=io.containerd.grpc.v1.content type=io.containerd.grpc.v1
time="2026-10-10T18:48:28.377727433Z" level=info msg="loading plugin" id=io.containerd.grpc.v1.diff type=io.containerd.grpc.v1
time="2026-10-10T18:48:28.377737163Z" level=info msg="loading plugin" id=io.containerd.grpc.v1.events type=io.containerd.grpc.v1
time="2026-10-10T18:48:28.377745923Z" level=info msg="loading plugin" id=io.containerd.grpc.v1.images type=io.containerd.grpc.v1
time="2026-10-10T18:48:28.377756111Z" level=info msg="loading plugin" id=io.containerd.grpc.v1.introspection type=io.containerd.grpc.v1
time="2026-10-10T18:48:28.377764774Z" level=info msg="loading plugin" id=io.containerd.grpc.v1.leases type=io.containerd.grpc.v1
time="2026-10-10T18:48:28.377773660Z" level=info msg="loading plugin" id=io.containerd.grpc.v1.mounts type=io.containerd.grpc.v1
time="2026-10-10T18:48:28.377782687Z" level=info msg="loading plugin" id=io.containerd.grpc

In [133]:
!dockerd \
    --iptables=false \
    --bridge=none \
    --ip-forward=false \
    --ip-masq=false \
    > /tmp/dockerd.log 2>&1 &

In [134]:
import time
time.sleep(10)

In [135]:
!docker info

Client:
 Version:    29.1.3
 Context:    default
 Debug Mode: false
 Plugins:
  trust: Manage trust on Docker images (Docker Inc.)
    Version:  29.1.3
    Path:     /usr/libexec/docker/cli-plugins/docker-trust

Server:
 Containers: 0
  Running: 0
  Paused: 0
  Stopped: 0
 Images: 0
 Server Version: 29.1.3
 Storage Driver: overlayfs
  driver-type: io.containerd.snapshotter.v1
 Logging Driver: json-file
 Cgroup Driver: cgroupfs
 Cgroup Version: 2
 Plugins:
  Volume: local
  Network: bridge host ipvlan macvlan null overlay
  Log: awslogs fluentd gcplogs gelf journald json-file local splunk syslog
 CDI spec directories:
  /etc/cdi
  /var/run/cdi
 Swarm: inactive
 Runtimes: io.containerd.runc.v2 runc
 Default Runtime: runc
 Init Binary: docker-init
 containerd version: 
 runc version: 
 init version: fec3683b971d9c3ef73f284f176672c44b448662
 Security Options:
  seccomp
   Profile: builtin
  cgroupns
 Kernel Version: 6.6.122+
 Operating System: Ubuntu 24.04.4 LTS
 OSType: linux
 Architectur

In [136]:
!docker run --rm hello-world

Unable to find image 'hello-world:latest' locally
latest: Pulling from library/hello-world


Digest: sha256:5e23090353324d887c48ad5e5c56d294eab81588df9605b07d1afe895f9cc8f8
Status: Downloaded newer image for hello-world:latest
docker: Error response from daemon: failed to mount /tmp/containerd-mount128998332: mount source: "overlay", target: "/tmp/containerd-mount128998332", fstype: overlay, flags: 0, data: "workdir=/var/lib/docker/containerd/daemon/io.containerd.snapshotter.v1.overlayfs/snapshots/2/work,upperdir=/var/lib/docker/containerd/daemon/io.containerd.snapshotter.v1.overlayfs/snapshots/2/fs,lowerdir=/var/lib/docker/containerd/daemon/io.containerd.snapshotter.v1.overlayfs/snapshots/1/fs,index=off", err: invalid argument

Run 'docker run --help' for more information


In [137]:
!docker info | grep -i "Storage Driver"

 Storage Driver: overlayfs


In [138]:
!docker run --rm hello-world

docker: Error response from daemon: failed to mount /tmp/containerd-mount2201540158: mount source: "overlay", target: "/tmp/containerd-mount2201540158", fstype: overlay, flags: 0, data: "workdir=/var/lib/docker/containerd/daemon/io.containerd.snapshotter.v1.overlayfs/snapshots/3/work,upperdir=/var/lib/docker/containerd/daemon/io.containerd.snapshotter.v1.overlayfs/snapshots/3/fs,lowerdir=/var/lib/docker/containerd/daemon/io.containerd.snapshotter.v1.overlayfs/snapshots/1/fs,index=off", err: invalid argument

Run 'docker run --help' for more information


In [139]:
!pkill dockerd || true
!pkill containerd || true

In [140]:
!rm -rf /tmp/docker-data
!mkdir -p /tmp/docker-data

In [141]:
!dockerd \
    --data-root=/tmp/docker-data \
    --storage-driver=vfs \
    --iptables=false \
    --bridge=none \
    --ip-forward=false \
    --ip-masq=false \
    > /tmp/dockerd.log 2>&1 &

In [142]:
import time
time.sleep(10)

In [143]:
!docker info | grep -E "Storage Driver|Docker Root Dir"

 Storage Driver: vfs
 Docker Root Dir: /tmp/docker-data


In [144]:
!docker run --rm hello-world

Unable to find image 'hello-world:latest' locally
latest: Pulling from library/hello-world

Digest: sha256:5e23090353324d887c48ad5e5c56d294eab81588df9605b07d1afe895f9cc8f8
Status: Downloaded newer image for hello-world:latest
docker: Error response from daemon: failed to create task for container: failed to create shim task: OCI runtime create failed: runc create failed: unable to start container process: unable to apply cgroup configuration: mkdir /sys/fs/cgroup/docker: read-only file system

Run 'docker run --help' for more information


In [145]:
!docker run --rm \
    --gpus all \
    --network host \
    nvidia/cuda:12.8.1-base-ubuntu24.04 \
    nvidia-smi

Unable to find image 'nvidia/cuda:12.8.1-base-ubuntu24.04' locally
12.8.1-base-ubuntu24.04: Pulling from nvidia/cuda





Digest: sha256:133c78a0575303be34164d0b90137a042172bdf60696af01a3c424ab402d86e2
Status: Downloaded newer image for nvidia/cuda:12.8.1-base-ubuntu24.04
docker: Error response from daemon: failed to create default sandbox: operation not permitted

Run 'docker run --help' for more information
